# Web Scrapping - Global Forest Watch data
- We create the output folder in case it does not exists in the "Downloads folder"
- This script iterates over the dataset repository
- Gets the link headers "xxxx.tif"
- Since they don't contain all the strcuture of the sentence, we build the input link with the link header
- Pointing to the download created folder we download them one by one

In [39]:
import requests
from bs4 import BeautifulSoup
import os
import re
from multiprocessing import Pool, cpu_count
import subprocess
from osgeo import gdal, osr

In [2]:
# URL of the website to scrape
url = "https://glad.umd.edu/users/Potapov/GLCLUC2020/Forest_height_2020/"
# Send a GET request to the URL
response = requests.get(url)
# Parse the HTML content
soup = BeautifulSoup(response.text, 'html.parser')
# Find all links in the webpage
links = soup.find_all('a')

In [43]:
# Create a folder inside "Downloads" to store downloaded files
def set_output_folder_in_download_folder(output_folder_name):
    output_admin_folder = "Downloads"
    download_folder = os.path.join(os.path.expanduser('~'), output_admin_folder, output_folder_name)
    if not os.path.exists(download_folder):
        os.makedirs(download_folder)
    print("Available the folder: " + output_folder_name)
    return download_folder

# Get the file links
def make_raster_link_list(links):
    web_file_list = []
    for link in links:
        href = link.get('href') # the name of the file
        if href and href.endswith('.tif'): # We would have to edit this
            web_file_list.append(href)
    return web_file_list

def download_file(file_url, file_location):
    retries = 3
    while retries > 0:
        failed_layers = []
        try:
            # Here we open the link and download the file
            with open(file_location, 'wb') as f:
                response = requests.get(file_url)
                f.write(response.content)
            break
        except Exception as e:
            print(f"Download failed for {file_location}. Retrying...")
            retries -= 1
            if retries == 0:
                print(f"Failed to download {file_location} after 3 attempts. Error: {str(e)}")
                failed_layers.append(file_location)
                break
    if failed_layers: # If there are failed layers, show them
        print("the next files were not downloaded")
        for file in failed_layers:
            print(file)

def gdalinfo_ok(filepath):
    """Run gdalinfo on a file. Returns (filepath, True/False)."""
    try:
        result = subprocess.run(
            ["gdalinfo", filepath],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=10  # seconds
        )
        return filepath, (result.returncode == 0)
    except:
        return filepath, False


def check_all_rasters_mp(folder, workers=None):
    """Check all TIFFs in a folder using multiprocessing."""
    tifs = [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(".tif")
    ]

    if workers is None:
        workers = max(1, cpu_count() - 1)

    with Pool(processes=workers) as pool:
        results = pool.map(gdalinfo_ok, tifs)

    valid = [f for f, ok in results if ok]
    invalid = [f for f, ok in results if not ok]

    return valid, invalid


def check_mosaic_consistency(folder):
    files = [f for f in os.listdir(folder) if f.lower().endswith(".tif")]
    if not files:
        print("No TIFF files found.")
        return

    reference = None
    errors = []

    for f in files:
        print(f)
        path = os.path.join(folder, f)
        ds = gdal.Open(path)
        if ds is None:
            errors.append(f"{f}: cannot be opened by GDAL")
            continue

        info = {
            "crs": ds.GetProjection(),
            "geotransform": ds.GetGeoTransform(),
            "pixel_size": (ds.GetGeoTransform()[1], ds.GetGeoTransform()[5]),
            "raster_size": (ds.RasterXSize, ds.RasterYSize),
            "bands": ds.RasterCount,
        }

        if reference is None:
            reference = info
            print(f"Using {f} as reference.")
        else:
            # Check CRS
            if info["crs"] != reference["crs"]:
                errors.append(f"{f}: CRS mismatch")

            # Check pixel resolution
            if info["pixel_size"] != reference["pixel_size"]:
                errors.append(f"{f}: pixel size mismatch")

            # Check geotransform cell rotation (should usually be zero)
            if (info["geotransform"][2], info["geotransform"][4]) != (0, 0):
                errors.append(f"{f}: rotation/shear in geotransform")

            # Check band count
            if info["bands"] != reference["bands"]:
                errors.append(f"{f}: band count mismatch")

    print("\n--- RESULTS ---")
    if errors:
        print("Inconsistencies found:")
        for e in errors:
            print(" -", e)
    else:
        print("All tiles are consistent.")


def check_and_fix_mosaic(folder):
    files = [f for f in os.listdir(folder) if f.lower().endswith(".tif")]
    if not files:
        print("No TIFF files found.")
        return

    reference = None
    errors = []

    for f in files:
        path = os.path.join(folder, f)
        print(f"Checking: {f}")
        ds = gdal.Open(path)
        if ds is None:
            errors.append(f"{f}: cannot be opened by GDAL")
            continue

        info = {
            "crs": ds.GetProjection(),
            "geotransform": ds.GetGeoTransform(),
            "pixel_size": (ds.GetGeoTransform()[1], ds.GetGeoTransform()[5]),
            "raster_size": (ds.RasterXSize, ds.RasterYSize),
            "bands": ds.RasterCount,
        }

        if reference is None:
            reference = info
            print(f"Using {f} as reference.")
            continue

        fix_needed = False

        # 1. CRS mismatch
        if info["crs"] != reference["crs"]:
            errors.append(f"{f}: CRS mismatch")
            fix_needed = True

        # 2. Pixel size mismatch
        if info["pixel_size"] != reference["pixel_size"]:
            errors.append(f"{f}: pixel size mismatch")
            fix_needed = True

        # 3. Rotation/shear
        if (info["geotransform"][2], info["geotransform"][4]) != (0, 0):
            errors.append(f"{f}: rotation/shear in geotransform")
            fix_needed = True

        # 4. Band count mismatch
        if info["bands"] != reference["bands"]:
            errors.append(f"{f}: band count mismatch")
            fix_needed = True

        # Fix tile if needed
        if fix_needed:
            fixed_path = os.path.join(folder, f"{os.path.splitext(f)[0]}_fixed.tif")
            print(f"Fixing tile: {f} -> {fixed_path}")

            # Reproject to reference CRS if needed
            target_srs = osr.SpatialReference()
            target_srs.ImportFromWkt(reference["crs"])

            gdal.Warp(
                fixed_path,
                path,
                dstSRS=target_srs,
                xRes=abs(reference["pixel_size"][0]),
                yRes=abs(reference["pixel_size"][1]),
                resampleAlg='near',
                multithread=True,
                creationOptions=[
                "TILED=YES",        # enable internal tiling
                "COMPRESS=LZW"      # use LZW compression
                ]
            )

    print("\n--- RESULTS ---")
    if errors:
        print("Inconsistencies found:")
        for e in errors:
            print(" -", e)
        print("\nTiles with issues have been fixed with '_fixed' suffix.")
    else:
        print("All tiles are consistent.")


In [12]:
output_folder_name = "NASA_forest_height_2020"
# Create the folder
download_folder = set_output_folder_in_download_folder(output_folder_name)

Available the folder: NASA_forest_height_2020


In [ ]:
web_file_list = []
if not links:
     # The input is expected to  be a set of links
     # Extract tifs using regular expressions
     links = re.findall(r'(https?://[^\s]+\.(?:tif|tiff))', response.text) # s?: 's' character optional due to the ? quantifier. / [^\s]: Any character except whitespace (\s) / + quantifier ensures that there is at least one non-whitespace character.
     for tif_url in links:
          filename = os.path.basename(tif_url)
          file_location = os.path.join(download_folder, filename) # location of the download
          web_file_list.append(filename)
          tif_response = requests.get(tif_url)
          print("Downloading: {}".format(filename))
          # download_file(tif_url, file_location)
else:
     # Download each treecover*.tif file into the created folder
    for link in links:
        href = link.get('href')  # the name of the file "kjdfnfosdjf.tif"
        if href and href.endswith('.tif'):
            file_location = os.path.join(download_folder, href) # location of the download
            web_file_list.append(href)
            file_url = url + href  # We build the sentence
            print("Downloading: ", file_location)
            download_file(file_url, file_location)

print("Download completed!")

In [ ]:
"""Check for missing files"""
# This works only if we use "href" / need to develop this further
if links:
    web_file_list = make_raster_link_list(links)
    print(len(web_file_list))
else: 
    web_file_list = links

# List all files in the download folder
local_file_list = os.listdir(download_folder)

# Create a mapping from lowercase -> original filename (for website files)
website_map = {f.lower(): f for f in web_file_list}

# Create lowercase sets
downloaded_files_set = {f.lower() for f in local_file_list}
website_files_set = set(website_map.keys())

# Missing lowercase names
missing_lower = website_files_set - downloaded_files_set

# Convert back to real names
missing_files = [website_map[m] for m in missing_lower]

if missing_files:
    print("The following files are missing:")
    for f in missing_files:
        print(f)
else:
    print("Everything is downloaded")


In [ ]:
"""Download missing files"""
layer_count = 1
for missing_file in missing_files:
    file_url = url + missing_file
    file_name = os.path.join(download_folder, missing_file)
    print("Downloading: ", file_name, "{} out of {}".format(layer_count, len(missing_files)))
    download_file(file_url, file_name)
    layer_count += 1
    
print("Missing files downloaded successfully!")

In [29]:
url = "https://forobs.jrc.ec.europa.eu/TMF/data#downloads"
resp = requests.get(url)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

titles = []
for div in soup.find_all("div", class_="tile"):
    title = div.get("title")
    if title:
        titles.append(title)

print(titles)


['N30_W110', 'N30_W100', 'N30_W90', 'N30_W80', 'N30_E70', 'N30_E80', 'N30_E90', 'N30_E100', 'N30_E110', 'N20_W110', 'N20_W100', 'N20_W90', 'N20_W80', 'N20_W70', 'N20_W60', 'N20_W20', 'N20_E70', 'N20_E80', 'N20_E90', 'N20_E100', 'N20_E110', 'N20_E120', 'N10_W90', 'N10_W80', 'N10_W70', 'N10_W60', 'N10_W50', 'N10_W20', 'N10_W10', 'N10_E0', 'N10_E10', 'N10_E20', 'N10_E30', 'N10_E40', 'N10_E70', 'N10_E80', 'N10_E90', 'N10_E100', 'N10_E110', 'N10_E120', 'N0_W90', 'N0_W80', 'N0_W70', 'N0_W60', 'N0_W50', 'N0_W40', 'N0_E0', 'N0_E10', 'N0_E20', 'N0_E30', 'N0_E40', 'N0_E90', 'N0_E100', 'N0_E110', 'N0_E120', 'N0_E130', 'N0_E140', 'N0_E150', 'N0_E160', 'S10_W80', 'S10_W70', 'S10_W60', 'S10_W50', 'S10_W40', 'S10_E10', 'S10_E20', 'S10_E30', 'S10_E40', 'S10_E50', 'S10_E120', 'S10_E130', 'S10_E140', 'S10_E150', 'S10_E160', 'S10_E170', 'S20_W70', 'S20_W60', 'S20_W50', 'S20_E40', 'S20_E50', 'S20_E140', 'S20_E150', 'S20_E160', 'S30_W60', 'S30_E140', 'S30_E150']


In [30]:

# Folder containing the .tif files
folder = r"C:\Users\admin\Downloads\RC_TMF_TransitionMap_Subtypes_v1_1982_2024"
# Extract tile codes that exist in your folder
found_tiles = set()

pattern = re.compile(r'(N|S)\d+_(E|W)\d+')

for fname in os.listdir(folder):
    match = pattern.search(fname)
    if match:
        found_tiles.add(match.group(0))

# Identify missing tiles
expected_set = set(titles)
missing_tiles = sorted(expected_set - found_tiles)

print("FOUND:", sorted(found_tiles))
print("\nMISSING:", missing_tiles)
print(f"\nTotal missing: {len(missing_tiles)}")

FOUND: ['N0_E0', 'N0_E10', 'N0_E100', 'N0_E110', 'N0_E120', 'N0_E130', 'N0_E140', 'N0_E150', 'N0_E160', 'N0_E20', 'N0_E30', 'N0_E40', 'N0_E90', 'N0_W40', 'N0_W50', 'N0_W60', 'N0_W70', 'N0_W80', 'N0_W90', 'N10_E0', 'N10_E10', 'N10_E100', 'N10_E110', 'N10_E120', 'N10_E20', 'N10_E30', 'N10_E40', 'N10_E70', 'N10_E80', 'N10_E90', 'N10_W10', 'N10_W20', 'N10_W50', 'N10_W60', 'N10_W70', 'N10_W80', 'N10_W90', 'N20_E100', 'N20_E110', 'N20_E120', 'N20_E70', 'N20_E80', 'N20_E90', 'N20_W100', 'N20_W110', 'N20_W20', 'N20_W60', 'N20_W70', 'N20_W80', 'N20_W90', 'N30_E100', 'N30_E110', 'N30_E70', 'N30_E80', 'N30_E90', 'N30_W100', 'N30_W110', 'N30_W80', 'N30_W90', 'S10_E10', 'S10_E120', 'S10_E130', 'S10_E140', 'S10_E150', 'S10_E160', 'S10_E170', 'S10_E20', 'S10_E30', 'S10_E40', 'S10_E50', 'S10_W40', 'S10_W50', 'S10_W60', 'S10_W70', 'S10_W80', 'S20_E140', 'S20_E150', 'S20_E160', 'S20_E40', 'S20_E50', 'S20_W50', 'S20_W60', 'S20_W70', 'S30_E140', 'S30_E150', 'S30_W60']

MISSING: []

Total missing: 0


In [45]:
check_mosaic_consistency(r"C:\Users\admin\Downloads\RC_TMF_TransitionMap_Subtypes_30m_v1_1982_2024_mosaic")

JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID16_S10_E10.tif
Using JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID16_S10_E10.tif as reference.
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID17_S10_E20.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID18_S10_E30.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID19_S10_E40.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID20_S10_E50.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID33_N0_E0.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID34_N0_E10.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID35_N0_E20.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID36_N0_E30.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID37_N0_E40.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID51_N10_W20.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID52_N10_W10.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID53_N10_E0.tif
JRC_TMF_TransitionMap_Subtypes_v1_1982_2024_AFR_ID54_N10_E10.tif
JRC_TMF_Tran

In [ ]:
check_and_fix_mosaic(r"C:\Users\admin\Downloads\RC_TMF_TransitionMap_Subtypes_30m_v1_1982_2024_mosaic")

In [49]:
folder = r"C:\Users\admin\Downloads\RC_TMF_TransitionMap_Subtypes_30m_v1_1982_2024_mosaic"
for f in os.listdir(folder):
    if f.lower().endswith(".tif"):
        path = os.path.join(folder, f)
        ds = gdal.Open(path)
        if ds is None:
            print(f"{f}: cannot be opened")
            continue

        gt = ds.GetGeoTransform()
        xsize = ds.RasterXSize
        ysize = ds.RasterYSize

        minx = gt[0]
        maxy = gt[3]
        maxx = gt[0] + gt[1]*xsize
        miny = gt[3] + gt[5]*ysize

        # print(f"{f}: {minx}, {miny}, {maxx}, {maxy}")

        # --- sanity checks ---
        errors = []

        # Longitude bounds
        if minx < -180 or maxx > 180:
            errors.append("longitude out of range")

        # Latitude bounds
        if miny < -90 or maxy > 90:
            errors.append("latitude out of range")

        # Negative width/height
        if maxx <= minx or maxy <= miny:
            errors.append("invalid bbox (negative width/height)")

        # Warn if tile crosses 180°
        if minx < -180 and maxx > 180:
            errors.append("crosses antimeridian")

        if errors:
            print(f"  WARNING: {f,', '.join(errors)}")